# Demo 07b — Voting

**Class 07 · E2** (slides: `slides/07_ensembles.md`)

The first ensembles need no new training algorithm: fit several models and let them vote.

* Why a crowd can win: the majority of independent voters and the variance of a mean.
* A vote of the six shallow models: hard, soft and weighted.
* k-NN with $k > 1$: a vote of the $k$ nearest training points, with equal or distance weights.
* The same figures as Demo 07a, with the new row added, and the failure cases of a vote.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from scipy import stats

from faa import demo, metrics, plotlib
from faa import galleries as ga
from faa import synthetic as sy
from faa.demo import show, slide
from faa.galleries import CLASS_SETS, MEMBERS, REG_SETS, board, family_figure, gain_figure, names_of
from faa.plotlib import BLUE, GREEN, GREY, ORANGE, PURPLE, RED, SKY

demo.setup("07")
SEED = 0
N_JOBS = 8

In [ ]:
slide("Wisdom of the Crowd: the Majority of M Voters")


def majority_accuracy(M: int, p: float) -> float:
    """P(more than half of M independent voters of accuracy p are right), M odd: a binomial sum."""
    return float(stats.binom.sf(M // 2, M, p))


p3 = 0.6
by_hand = p3**3 + 3 * p3**2 * (1 - p3)
print(
    f"M = 3, p = {p3}: p^3 + 3 p^2 (1 - p) = {p3**3:.3f} + {3 * p3**2 * (1 - p3):.3f} = {by_hand:.3f}; binomial sum {majority_accuracy(3, p3):.3f}"
)
rows = [(M, *[majority_accuracy(M, p) for p in (0.45, 0.5, 0.55, 0.6, 0.7)]) for M in (1, 3, 5, 11, 21, 51, 101)]
show(pl.DataFrame(rows, schema=["M", "p=0.45", "p=0.50", "p=0.55", "p=0.60", "p=0.70"], orient="row"))
print("members with p < 0.5 make the crowd worse: the same sum goes to 0 as M grows; p = 0.5 stays at 0.5")

In [ ]:
slide("Voters That Copy Each Other")


def correlated_majority(M: int, p: float, c: float) -> float:
    """One-factor model: with probability c all M voters copy one common answer (right with probability p),
    otherwise they are independent. Accuracy of the majority = c p + (1 - c) majority_accuracy(M, p)."""
    return c * p + (1 - c) * majority_accuracy(M, p)


rng = np.random.default_rng(SEED)
M_CHECK, N_MC, C_CHECK = 11, 200_000, 0.5
common = rng.random(N_MC) < 0.6
indep = rng.random((N_MC, M_CHECK)) < 0.6
copy = rng.random(N_MC) < C_CHECK
votes = np.where(copy[:, None], common[:, None], indep)
print(
    f"Monte Carlo check, M = {M_CHECK}, p = 0.6, c = {C_CHECK}: simulated {np.mean(votes.sum(axis=1) > M_CHECK / 2):.4f}, formula {correlated_majority(M_CHECK, 0.6, C_CHECK):.4f}"
)
rows = [
    (c, correlated_majority(1, 0.6, c), correlated_majority(11, 0.6, c), correlated_majority(101, 0.6, c))
    for c in (0.0, 0.25, 0.5, 0.75, 1.0)
]
show(pl.DataFrame(rows, schema=["copy share c", "M = 1", "M = 11", "M = 101"], orient="row"))
print("c = 1: identical members, the crowd is one voter and M does not matter")

In [ ]:
# plot e2_condorcet
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), sharey=True)
Ms = np.arange(1, 102, 2)
for p, col in ((0.7, GREEN), (0.6, BLUE), (0.55, SKY), (0.5, GREY), (0.45, RED)):
    y_end = [majority_accuracy(int(M), p) for M in Ms]
    axs[0].plot(Ms, y_end, color=col)
    axs[0].text(103, {0.7: 1.04, 0.6: 0.95}.get(p, y_end[-1]), f"$p$ = {p}", color=col, fontsize=6.3, va="center")
for c, col in ((0.0, BLUE), (0.25, SKY), (0.5, ORANGE), (0.75, PURPLE), (1.0, RED)):
    y_end = [correlated_majority(int(M), 0.6, c) for M in Ms]
    axs[1].plot(Ms, y_end, color=col)
    axs[1].text(103, y_end[-1], f"$c$ = {c}", color=col, fontsize=6.3, va="center")
axs[0].set(xlabel="voters $B$, independent", ylabel="P(the majority is right)", ylim=(0, 1.02), xlim=(0, 135))
axs[1].set(xlabel="voters $B$, $p$ = 0.6, share $c$ copy", xlim=(0, 135))
fg.subplots_adjust(wspace=0.08)
plotlib.save(fg, "e2_condorcet")

In [ ]:
slide("The Variance of an Average of Correlated Errors")
SIGMA = 1.0


def var_of_mean(S: int, rho: float, sigma: float = SIGMA) -> float:
    """Variance of the mean of S errors of variance sigma^2 and pairwise correlation rho: rho s^2 + (1 - rho) s^2 / S."""
    return rho * sigma**2 + (1 - rho) * sigma**2 / S


rng = np.random.default_rng(SEED)
rows = []
for rho in (0.0, 0.3, 0.6, 1.0):
    for S in (1, 5, 50):
        shared = rng.normal(size=(100_000, 1))
        own = rng.normal(size=(100_000, S))
        e = np.sqrt(rho) * shared + np.sqrt(1 - rho) * own  # variance 1, pairwise correlation rho
        rows.append((rho, S, var_of_mean(S, rho), float(e.mean(axis=1).var())))
show(pl.DataFrame(rows, schema=["rho", "S", "formula", "simulated"], orient="row"))
print("S -> infinity: the variance goes to rho sigma^2: more members cannot remove the shared part of the error")

In [ ]:
# plot e2_variance
fg, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
Ss = np.arange(1, 51)
for rho, col in ((0.0, BLUE), (0.3, GREEN), (0.6, ORANGE), (1.0, RED)):
    ax.plot(Ss, [var_of_mean(int(S), rho) for S in Ss], color=col, label=f"$\\rho$ = {rho}")
ax.set(xlabel="members $B$", ylabel="variance of the mean / $\\sigma^2$", ylim=(0, 1.05), xlim=(1, 50))
ax.legend(ncols=2, loc="upper right", fontsize=6.3, handlelength=1.2, columnspacing=0.8, labelspacing=0.2)
plotlib.save(fg, "e2_variance")

In [ ]:
slide("The Error of the Mean")
X_tr, y_tr, X_te, y_te = sy.load("sawtooth", SEED)
F = np.array([m.fit(X_tr, y_tr).predict(X_te) for m in ga.shallow_regressors(SEED).values()])
f_bar = F.mean(axis=0)
mean_mse, ambiguity = float(np.mean((F - y_te) ** 2)), float(np.mean((F - f_bar) ** 2))
print(
    f"B = {len(F)} members on the sawtooth: mean member MSE {mean_mse:.4f} - ambiguity {ambiguity:.4f} = {mean_mse - ambiguity:.4f}; MSE of the mean {np.mean((f_bar - y_te) ** 2):.4f}"
)

## Three votes of the six shallow models

**Hard vote:** each member names a class. **Soft vote:** the mean of the probabilities. **Weighted vote:** a soft vote where each member counts with its cross-validated MCC on the training set, floored at 0, so a member no better than chance has no say. In regression: the median, the mean and a mean weighted by the inverse of the MAE.

In [ ]:
slide("Weights of the Vote")
X_tr, y_tr, X_te, y_te = sy.load("spiral", SEED)
vote = ga.WeightedVote(ga.shallow_classifiers(SEED), "c").fit(X_tr, y_tr)
total = sum(vote.weights_.values())
show(
    pl.DataFrame(
        {
            "member": list(vote.weights_),
            "CV MCC": list(vote.weights_.values()),
            "share of the vote": [w / total for w in vote.weights_.values()],
        }
    )
)

In [ ]:
# plot e2_weights
fg, ax = plotlib.fig("full", 1.15)
names, shares = list(vote.weights_), [w / total for w in vote.weights_.values()]
ax.bar(names, shares, color=BLUE, width=0.6)
ax.axhline(1 / len(names), color=GREY, lw=0.7, ls=":")
for i, v in enumerate(shares):
    ax.text(i, v + 0.008, f"{v:.3f}", ha="center", fontsize=6.2)
ax.set(ylabel="share of the vote", ylim=(0, max(shares) * 1.25))
ax.grid(axis="x", visible=False)
plotlib.save(fg, "e2_weights")

In [ ]:
slide("Voting: the Board")
F = ["shallow", "voting"]
clf = ga.evaluate(CLASS_SETS, names_of("c", F))
show(ga.wide(clf, names_of("c", F), CLASS_SETS))

In [ ]:
reg = ga.evaluate(REG_SETS, names_of("r", F))
show(ga.wide(reg, names_of("r", F), REG_SETS))

In [ ]:
gain_c = ga.gain(clf, CLASS_SETS, ga.VOTING_C, ga.SHALLOW_C, "c")
gain_r = ga.gain(reg, REG_SETS, ga.VOTING_R, ga.SHALLOW_R, "r")
show(gain_c)
show(gain_r)
vc = ga.gain(clf, CLASS_SETS, ("hard vote", "soft vote", "weighted vote"), ga.SHALLOW_C, "c")
print("the three votes of the six models, without k-NN:")
show(vc)

In [ ]:
slide("k-NN Is a Vote")


def k_run(dataset, k, seed):
    X_tr, y_tr, X_te, y_te = sy.load(dataset, seed)
    model = ga.build("c", "15-NN", seed).set_params(kneighborsclassifier__n_neighbors=k).fit(X_tr, y_tr)
    return dataset, k, metrics.task_score("c", y_te, model.predict(X_te))


KS = (1, 3, 5, 9, 15, 25, 51, 101)
rows = Parallel(n_jobs=N_JOBS)(
    delayed(k_run)(d, k, s) for d in ("spiral", "moons", "checkerboard") for k in KS for s in ga.SEEDS
)
sweep = (
    pl.DataFrame(rows, schema=["dataset", "k", "MCC"], orient="row").group_by("dataset", "k").agg(pl.col("MCC").mean())
)
show(sweep.pivot(on="dataset", index="k", values="MCC").sort("k"))

In [ ]:
# plot e2_knn
fg, ax = plotlib.fig("full", 1.35)
for d, col in (("spiral", BLUE), ("moons", ORANGE), ("checkerboard", GREEN)):
    s = sweep.filter(pl.col("dataset") == d).sort("k")
    ax.plot(s["k"], s["MCC"], color=col, marker="o", ms=2.5, label=d)
ax.axhline(1 - 2 * sy.FLIP, color=GREY, lw=0.7, ls=":")
ax.set(xscale="log", xlabel="neighbours $k$ (log scale)", ylabel="test MCC", xticks=KS, xticklabels=KS)
ax.legend(ncols=3, loc="lower center", fontsize=6.3, handlelength=1.4)
plotlib.save(fg, "e2_knn")

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C}
plotlib.save(family_figure("spiral", rows, {"voting": MEMBERS["c"]}), "e2_gallery_spiral")

In [ ]:
rows = {"shallow": ga.SHALLOW_C, "voting": ga.VOTING_C}
plotlib.save(family_figure("checkerboard", rows, {"voting": MEMBERS["c"]}), "e2_gallery_checker")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R}
plotlib.save(family_figure("sawtooth", rows, {"voting": MEMBERS["r"]}), "e2_gallery_saw")

In [ ]:
rows = {"shallow": ga.SHALLOW_R, "voting": ga.VOTING_R}
plotlib.save(family_figure("chirp", rows, {"voting": MEMBERS["r"]}), "e2_gallery_chirp")

In [ ]:
plotlib.save(board(clf, "c", CLASS_SETS), "e2_board_c")

In [ ]:
plotlib.save(board(reg, "r", REG_SETS), "e2_board_r")

In [ ]:
plotlib.save(gain_figure(gain_c, "gain in MCC: best voter over best shallow model"), "e2_gain_c")

In [ ]:
plotlib.save(gain_figure(gain_r, "gain in MAE: best voter over best shallow model"), "e2_gain_r")

## When a vote fails (no slide)

Weak members can outvote a good one, identical members add nothing, members with one bias keep it.

In [ ]:
slide("When a Vote Fails (no slide)")
from sklearn.base import clone  # noqa: E402
from sklearn.ensemble import VotingClassifier  # noqa: E402

base = ga.shallow_classifiers(SEED)


def fail(title, dataset, names):
    X_tr, y_tr, X_te, y_te = sy.load(dataset, SEED)
    mem = [(f"{n} {i}", clone(base[n])) for i, n in enumerate(names)]
    out = [(n, metrics.task_score("c", y_te, m.fit(X_tr, y_tr).predict(X_te))) for n, m in mem]
    for kind in ("hard", "soft"):
        out.append(
            (
                f"{kind} vote",
                metrics.task_score("c", y_te, VotingClassifier(mem, voting=kind).fit(X_tr, y_tr).predict(X_te)),
            )
        )
    print(f"{title} ({dataset}):")
    show(pl.DataFrame(out, schema=["model", "MCC"], orient="row"))


fail("two weak members against a good one", "XOR", ["logistic", "naive Bayes", "SVM RBF"])
fail("identical members", "moons", ["SVM RBF", "SVM RBF", "SVM RBF"])
fail("members with one bias", "circles", ["logistic", "logistic", "logistic"])